In [143]:
import torch
print(torch.cuda.is_available())

True


In [144]:
# 셀 0: 작업 디렉토리 구조 생성
import os
from pathlib import Path

# 작업 루트 설정 (홈 기준)
WORKSPACE = Path.home() / "workspace"

dirs = [
    WORKSPACE / "notebooks",
    WORKSPACE / "src",
    WORKSPACE / "data" / "files",      # 여기에 RFP 문서 넣기
    WORKSPACE / "data" / "metadata",   # data_list.csv 여기
    WORKSPACE / "artifacts",           # 산출물 저장
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"✓ {d}")

print("\n작업 디렉토리 구조 생성 완료!")
print(f"작업 루트: {WORKSPACE}")

✓ /home/spai0703/workspace/notebooks
✓ /home/spai0703/workspace/src
✓ /home/spai0703/workspace/data/files
✓ /home/spai0703/workspace/data/metadata
✓ /home/spai0703/workspace/artifacts

작업 디렉토리 구조 생성 완료!
작업 루트: /home/spai0703/workspace


In [145]:
# 셀 1: 패키지 설치 (myenv 기준)
import sys
import os

packages = [
    "pypdf",
    "pdfplumber",
    "python-docx",
    "pandas",
    "numpy",
    "tqdm",
    "python-dotenv",
    "openai",
    "tiktoken",
    "gdown",
    # LangChain + Chroma (FAISS 대신)
    "langchain",
    "langchain-openai",
    "langchain-community",
    "langchain-chroma",
    "chromadb",
    # Hybrid Search
    "rank-bm25",
    # OCR
    "easyocr",
    "Pillow",
    "pdf2image",   # PDF → 이미지 변환 (OCR 전처리)
]

for pkg in packages:
    result = os.system(f"{sys.executable} -m pip install {pkg} -q")
    status = "✓" if result == 0 else "✗"
    print(f"{status} {pkg}")

print("\n설치 완료!")

✓ pypdf
✓ pdfplumber
✓ python-docx
✓ pandas
✓ numpy
✓ tqdm
✓ python-dotenv
✓ openai
✓ tiktoken
✓ gdown
✓ langchain
✓ langchain-openai
✓ langchain-community
✓ langchain-chroma
✓ chromadb
✓ rank-bm25
✓ easyocr
✓ Pillow
✓ pdf2image

설치 완료!


In [146]:
# 셀 2: 환경변수 설정
# ~/workspace/.env 파일 생성 후 아래처럼 사용

from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.home() / "workspace" / ".env"

# .env 파일이 없으면 템플릿 생성
if not env_path.exists():
    env_path.write_text('OPENAI_API_KEY=여기에_API_키_입력\n')
    print(f".env 파일 생성됨: {env_path}")
    print("⚠️  API 키를 직접 입력해주세요!")
else:
    print(f".env 파일 확인됨: {env_path}")

load_dotenv(env_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY and OPENAI_API_KEY != "여기에_API_키_입력":
    print(f"✓ API 키 로드 완료: {OPENAI_API_KEY[:8]}...")
else:
    print("⚠️  .env 파일에 OPENAI_API_KEY를 입력해주세요")

.env 파일 확인됨: /home/spai0703/workspace/.env
✓ API 키 로드 완료: sk-proj-...


In [147]:
from pathlib import Path
import zipfile

WORKSPACE = Path.home() / "workspace"
files_dir = WORKSPACE / "data" / "files"

# 업로드된 zip 경로 (JupyterHub로 직접 업로드한 파일)
zip_out = WORKSPACE / "data" / "files" / "codeit_project.zip"

# 파일 존재 확인
import os
if not zip_out.exists():
    print(f"⚠️  파일 없음: {zip_out}")
else:
    size = os.path.getsize(zip_out)
    print(f"✓ 파일 확인: {size / 1024 / 1024:.1f} MB")

    # zip 검증 후 압축 해제
    if zipfile.is_zipfile(zip_out):
        with zipfile.ZipFile(zip_out, "r") as zf:
            zf.extractall(str(files_dir))
        
        # zip 파일 자체 제외하고 실제 문서만 카운트
        extracted = [f for f in files_dir.glob("*") if f.name != "codeit_project.zip"]
        print(f"✓ 압축 해제 완료: {len(extracted)}개 파일")
        print(f"  예시: {[f.name for f in extracted[:5]]}")
    else:
        print("⚠️  유효한 zip 파일이 아닙니다")

✓ 파일 확인: 150.0 MB
✓ 압축 해제 완료: 2개 파일
  예시: ['.ipynb_checkpoints', 'files']


In [148]:
# 셀 3: 전역 경로 및 설정값
from pathlib import Path
import os

# ── 경로 ──────────────────────────────────────────
WORKSPACE    = Path.home() / "workspace"
FILES_DIR    = WORKSPACE / "data" / "files"
METADATA_CSV = WORKSPACE / "data" / "metadata" / "data_list.csv"
ARTIFACTS    = WORKSPACE / "artifacts"

DOCUMENT_INDEX_JSON = ARTIFACTS / "document_index.json"  # CSV → JSON으로 변경
CHUNKS_JSON         = ARTIFACTS / "chunks.json"
EMBEDDINGS_NPY      = ARTIFACTS / "embeddings.npy"

# ── 모델 설정 ──────────────────────────────────────
GEN_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

# ── 청킹 설정 ──────────────────────────────────────
CHUNK_MAX_CHARS = 1200
CHUNK_OVERLAP   = 150
RETRIEVAL_TOP_K = 5

# ── artifacts 폴더 생성 (없으면 자동 생성) ──────────
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# ── 경로 확인 ──────────────────────────────────────
print("경로 설정:")
for name, path in [
    ("FILES_DIR",    FILES_DIR),
    ("METADATA_CSV", METADATA_CSV),
    ("ARTIFACTS",    ARTIFACTS),
]:
    exists = "✓" if path.exists() else "✗ (없음)"
    print(f"  {exists} {name}: {path}")

경로 설정:
  ✓ FILES_DIR: /home/spai0703/workspace/data/files
  ✓ METADATA_CSV: /home/spai0703/workspace/data/metadata/data_list.csv
  ✓ ARTIFACTS: /home/spai0703/workspace/artifacts


In [149]:
# 셀 4: 데이터 확인
import pandas as pd
import unicodedata
import re
from pathlib import Path

# ── 파일명 정규화 함수 (전역 사용) ────────────────────
def normalize_fname(name: str) -> str:
    """파일명 정규화: NFKC + 공백 종류 통일"""
    name = unicodedata.normalize("NFKC", name)
    name = name.strip()
    name = re.sub(r"[\s\u00A0\u3000]+", " ", name)
    return name

# ── FILES_DIR 실제 파일 위치 자동 탐색 ──────────────
def resolve_files_dir(base_dir: Path) -> Path:
    """
    실제 RFP 파일이 있는 폴더를 찾아서 반환.
    zip 해제 후 files/files/ 구조로 중첩된 경우 자동 처리.
    """
    exts = {".pdf", ".hwp", ".hwpx", ".docx"}

    direct = [f for f in base_dir.glob("*") if f.is_file() and f.suffix.lower() in exts]
    if len(direct) >= 5:
        return base_dir

    for sub in base_dir.rglob("*"):
        if sub.is_dir() and sub.name != ".ipynb_checkpoints":
            sub_files = [f for f in sub.glob("*") if f.is_file() and f.suffix.lower() in exts]
            if len(sub_files) >= 5:
                print(f"  → 실제 파일 위치 감지: {sub}")
                return sub
    return base_dir

ACTUAL_FILES_DIR = resolve_files_dir(FILES_DIR)

# ── 파일 수 확인 ─────────────────────────────────
hwp_count = len(list(ACTUAL_FILES_DIR.glob("*.hwp")))
pdf_count = len(list(ACTUAL_FILES_DIR.glob("*.pdf")))
docx_count = len(list(ACTUAL_FILES_DIR.glob("*.docx")))
total = hwp_count + pdf_count + docx_count

print(f"FILES_DIR: {ACTUAL_FILES_DIR}")
print(f"  HWP: {hwp_count}개 / PDF: {pdf_count}개 / DOCX: {docx_count}개 / 총: {total}개")

# ── CSV 로드 ──────────────────────────────────────
df = pd.read_csv(METADATA_CSV)
df.columns = [c.strip() for c in df.columns]
print(f"\nCSV 행 수: {len(df)}")
print(f"컬럼 목록: {list(df.columns)}\n")

# ── 파일 형식 분포 ─────────────────────────────────
if "파일형식" in df.columns:
    print("파일 형식 분포:")
    print(df["파일형식"].value_counts())
    print()

# ── 파일 존재 여부 확인 (정규화 매칭) ─────────────────
if "파일명" in df.columns:
    # 실제 파일명을 정규화 → 원본 매핑
    actual_norm_map = {
        normalize_fname(f.name): f.name
        for f in ACTUAL_FILES_DIR.glob("*") if f.is_file()
    }

    found, missing = 0, []
    for fname in df["파일명"].dropna():
        norm = normalize_fname(str(fname).strip())
        if norm in actual_norm_map:
            found += 1
        else:
            missing.append(str(fname).strip())

    print(f"파일 확인: {found}개 존재 / {len(missing)}개 없음")
    if missing:
        print(f"  없는 파일 예시: {missing[:3]}")

# ── 텍스트 컬럼 품질 확인 ──────────────────────────
if "텍스트" in df.columns:
    text_lens = df["텍스트"].fillna("").apply(len)
    print(f"\n텍스트 컬럼 길이:")
    print(f"  평균: {text_lens.mean():.0f}자")
    print(f"  최소: {text_lens.min()}자")
    print(f"  최대: {text_lens.max()}자")
    print(f"  빈 값: {(text_lens == 0).sum()}개")

# ── FILES_DIR 전역 업데이트 ────────────────────────
FILES_DIR = ACTUAL_FILES_DIR
print(f"\n✓ FILES_DIR 업데이트 완료: {FILES_DIR}")
print(f"✓ normalize_fname() 전역 사용 가능")
df.head(3)

  → 실제 파일 위치 감지: /home/spai0703/workspace/data/files/files
FILES_DIR: /home/spai0703/workspace/data/files/files
  HWP: 96개 / PDF: 4개 / DOCX: 1개 / 총: 101개

CSV 행 수: 100
컬럼 목록: ['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트']

파일 형식 분포:
파일형식
hwp    96
pdf     4
Name: count, dtype: int64

파일 확인: 100개 존재 / 0개 없음

텍스트 컬럼 길이:
  평균: 3844자
  최소: 89자
  최대: 18335자
  빈 값: 0개

✓ FILES_DIR 업데이트 완료: /home/spai0703/workspace/data/files/files
✓ normalize_fname() 전역 사용 가능


,공고 번호,공고 차수,사업명,사업 금액,발주 기관,공개 일자,입찰 참여 시작일,입찰 참여 마감일,사업 요약,파일형식,파일명,텍스트
0,20241001798,0.0,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화,130000000.0,한영대학,2024-10-04 13:51:23,NaN,2024-10-15 17:00:00,- 한영대학교 특성화 맞춤형 교육환경 구축을 위해 트랙운영 학사정보시스템을 고도화한...,hwp,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,\n \n2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 ...
1,20241002912,0.0,2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선,129300000.0,한국연구재단,2024-10-04 15:01:52,2024-10-14 10:00:00,2024-10-16 14:00:00,- 사업 개요: 2024년 대학 산학협력활동 실태조사 시스템(UICC) 기능개선\n...,hwp,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,\r\n \r\n \r\n \r\n제 안 요 청 서\r\n[ 2024년 대학 ...
2,20240827859,0.0,EIP3.0 고압가스 안전관리 시스템 구축 용역,40000000.0,한국생산기술연구원,2024-08-28 11:31:02,2024-08-29 09:00:00,2024-09-09 10:00:00,- 사업 개요: EIP3.0 고압가스 안전관리 시스템 구축 용역\n- 추진배경: 안...,hwp,한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp,\r\n \r\nEIP3.0 고압가스 안전관리\r\n시스템 구축 용역\...


In [150]:
# 셀 5: 파서 — PDF + DOCX + HWP + OCR + 특수문자 정제
import pdfplumber
from pypdf import PdfReader
from docx import Document as DocxDocument
from pathlib import Path
from PIL import Image
import re
import io
import subprocess
import sys
import pyhwp2md

# ── OCR 초기화 (최초 1회만 로드) ──────────────────────
_ocr_reader = None

def get_ocr_reader():
    global _ocr_reader
    if _ocr_reader is None:
        import easyocr
        import torch
        use_gpu = torch.cuda.is_available()
        print(f"  OCR 초기화 중... (GPU: {use_gpu})")
        _ocr_reader = easyocr.Reader(["ko", "en"], gpu=use_gpu)
        print("  ✓ OCR 준비 완료")
    return _ocr_reader


def clean_text(text: str) -> str:
    text = re.sub(r"-\s*\d+\s*-", "", text)
    text = re.sub(r"[□■▣◇◆△▽](?!\s*\w)", " ", text)
    text = re.sub(r"[·•]{2,}", "•", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_query_text(text: str) -> str:
    """
    검색용 텍스트 정규화: 띄어쓰기 제거 버전 생성
    BM25 키워드 매칭 정확도 향상용
    """
    return re.sub(r"\s+", "", text)


def table_to_markdown(table: list, page_num: int = 0, label: str = "") -> str:
    if not table:
        return ""
    md_rows = []
    for i, row in enumerate(table):
        cells = [str(c).strip().replace("\n", " ") if c else "" for c in row]
        md_rows.append("| " + " | ".join(cells) + " |")
        if i == 0:
            md_rows.append("| " + " | ".join(["---"] * len(cells)) + " |")
    prefix = label or f"[표 p{page_num+1}]"
    return prefix + "\n" + "\n".join(md_rows)


def ocr_pdf_page(page, page_num: int) -> str:
    try:
        reader = get_ocr_reader()
        img = page.to_image(resolution=200).original
        img_bytes = io.BytesIO()
        img.save(img_bytes, format="PNG")
        img_bytes.seek(0)
        img_array = Image.open(img_bytes)
        results = reader.readtext(img_array, detail=0, paragraph=True)
        ocr_text = "\n".join(results)
        return f"[OCR p{page_num+1}]\n{ocr_text}" if ocr_text.strip() else ""
    except Exception as e:
        print(f"  [OCR 오류] p{page_num+1}: {e}")
        return ""


def extract_text_from_pdf(file_path: str, use_ocr: bool = True) -> str:
    page_contents = []
    try:
        reader = PdfReader(file_path)
        pypdf_pages = [p.extract_text() or "" for p in reader.pages]
    except Exception as e:
        print(f"[pypdf 오류] {Path(file_path).name}: {e}")
        pypdf_pages = []

    try:
        with pdfplumber.open(file_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                parts = []
                base_text = pypdf_pages[page_num] if page_num < len(pypdf_pages) else ""

                if use_ocr and len(base_text.strip()) < 50:
                    ocr_result = ocr_pdf_page(page, page_num)
                    if ocr_result:
                        parts.append(ocr_result)
                    elif base_text.strip():
                        parts.append(base_text)
                else:
                    if base_text.strip():
                        parts.append(base_text)

                tables = page.extract_tables()
                for t_idx, table in enumerate(tables):
                    if not table:
                        continue
                    table_flat = " ".join(str(c) for row in table for c in row if c)
                    is_schedule = any(
                        kw in table_flat
                        for kw in ["M1", "M2", "M3", "월", "분기", "착수", "완료", "시행"]
                    )
                    label = "[추진일정표 — 색상 기반 일정은 텍스트 추출 불가, 구조만 보존]\n" \
                            if is_schedule else ""
                    md = table_to_markdown(table, page_num, label)
                    if md:
                        parts.append(md)

                if parts:
                    page_contents.append("\n\n".join(parts))
    except Exception as e:
        print(f"[pdfplumber 오류] {Path(file_path).name}: {e}")
        page_contents = [clean_text(t) for t in pypdf_pages if t.strip()]

    return clean_text("\n\n".join(page_contents))


def extract_text_from_docx(file_path: str) -> str:
    try:
        doc = DocxDocument(file_path)
        contents = []
        for p in doc.paragraphs:
            if p.text.strip():
                contents.append(p.text)
        for t_idx, table in enumerate(doc.tables):
            md = table_to_markdown(
                [[c.text for c in row.cells] for row in table.rows],
                label=f"[표{t_idx+1}]"
            )
            if md:
                contents.append(md)
        return clean_text("\n\n".join(contents))
    except Exception as e:
        print(f"[DOCX 파싱 오류] {Path(file_path).name}: {e}")
        return ""


# ── HWP 파서 (pyhwp 대신 pyhwp2md 설치) ──────────────────────
def ensure_hwp_parser():
    """pyhwp2md 설치 확인"""
    try:
        import pyhwp2md
        print("  ✓ pyhwp2md 이미 설치됨")
    except ImportError:
        print("  pyhwp2md 설치 중...")
        import subprocess
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "pyhwp2md", "-q"],
            capture_output=True
        )
        print("  ✓ pyhwp2md 설치 완료")


def extract_text_from_hwp(file_path: str) -> str:
    """
    HWP 파서: pyhwp2md + CSV 텍스트 결합
    - pyhwp2md: 표 구조 추출 (Markdown)
    - CSV 텍스트: 본문 내용 보완
    """
    parts = []
    
    # 1) pyhwp2md로 표 추출
    try:
        from pyhwp2md import convert
        md_text = convert(str(file_path))
        if md_text and md_text.strip():
            parts.append(md_text.strip())
    except Exception as e:
        print(f"  [HWP pyhwp2md 오류] {Path(file_path).name}: {e}")
    
    combined = "\n\n".join(parts)
    return clean_text(combined) if combined.strip() else ""


def extract_text_from_file(file_path: str, use_ocr: bool = True) -> str:
    suffix = Path(file_path).suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(file_path, use_ocr=use_ocr)
    elif suffix == ".docx":
        return extract_text_from_docx(file_path)
    elif suffix in [".hwp", ".hwpx"]:
        return extract_text_from_hwp(file_path)
    return ""


# ── 설치 + 테스트 ────────────────────────────────────
ensure_hwp_parser()

# HWP 테스트
hwp_files = list(FILES_DIR.glob("*.hwp")) + list(FILES_DIR.glob("*.hwpx"))
if hwp_files:
    sample_hwp = hwp_files[0]
    print(f"\nHWP 파싱 테스트: {sample_hwp.name}")
    hwp_text = extract_text_from_hwp(str(sample_hwp))
    if hwp_text:
        print(f"  ✓ 텍스트: {len(hwp_text)}자")
        print(f"  표 감지: {hwp_text.count('|')}개 셀")
        print(f"\n--- 앞 500자 ---\n{hwp_text[:500]}")
    else:
        print(f"  ⚠️ HWP 파싱 실패 — CSV fallback 사용")
else:
    print("\nHWP 파일 없음")

  ✓ pyhwp2md 이미 설치됨

HWP 파싱 테스트: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp
  ✓ 텍스트: 5060자
  표 감지: 51개 셀

--- 앞 500자 ---
| 「개인정보 안전성 확보조치 기준 고시」(개인정보보호위원회 고시 제2023-6호) 및 「개인정보 보호법」 제28조에 따라 개인정보처리자 및 취급자는 개인정보보호에 관한 교육을 의무적으로 시행하여야 한다. | ④ 제1항에 따른 교육의 시기와 방법 등에 대해서는 “갑”은 “을”과 협의하여 시행한다. | 제9조 (정보주체 권리보장) ① “을”은 정보주체의 개인정보 열람, 정정·삭제, 처리 정지 요청 등에 대응하기 위한 연락처 등 민원 창구를 마련해야 한다. | 제10조 (개인정보의 파기) ① “을”은 제4항의 위탁업무기간이 종료되면 특별한 사유가 없는 한 지체 없이 개인정보를 파기하고 이를 “갑”에게 확인받아야 한다. | 제11조 (손해배상) ① “을” 또는 “을”의 임직원 기타 “을”의 수탁자가 이 계약에 의하여 위탁 또는 재위탁받은 업무를 수행함에 있어 이 계약에 따른 의무를 위반하거나 “을” 또는 “을”의 임직원 기타 “을”의 수탁자의 귀책사유로 인하여 이 계약이 해지되어 “갑” 


In [151]:
# ============================================================
# 셀 6: 항목 단위 세분화 청킹 (RFP 특화)
# ============================================================
import re
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any

# ── 청킹 설정 (조정) ─────────────────────────────────
CHUNK_MAX_CHARS = 800    # ← 1200→800 (세분화했으므로 줄임)
CHUNK_OVERLAP = 100      # ← 150→100

# ── 2단계 섹션 패턴: 대분류 + 소분류 ─────────────────

# 대분류 (기존과 동일)
SECTION_PATTERNS = {
    "overview":          ["사업개요", "제안요청개요", "입찰개요", "과업개요", "사업안내"],
    "background":        ["추진배경", "사업배경", "추진배경및필요성", "필요성", "현황및문제점"],
    "objective":         ["추진목표", "사업목적", "목적"],
    "scope":             ["과업내용", "사업내용", "주요사업내용", "수행범위", "과업범위"],
    "eligibility":       ["입찰참가자격", "참가자격", "지원자격", "신청자격"],
    "submission_docs":   ["제출서류", "구비서류", "신청서류", "입찰등록서류"],
    "submission_method": ["제출방법", "접수방법", "제출처", "접수처"],
    "deadline":          ["제출마감", "마감일", "접수마감", "입찰마감", "일정"],
    "evaluation":        ["평가기준", "평가방법", "선정기준", "제안서평가"],
    "contract":          ["계약방법", "계약조건", "계약체결"],
    "budget":            ["사업금액", "사업예산", "추정금액", "예산"],
    "requirements":      ["요구사항", "기능요구사항", "기술요구사항", "시스템요구사항"],
    "schedule":          ["추진일정", "사업일정", "수행일정"],
    "toc":               ["목차"],
    "etc":               ["유의사항", "기타사항", "참고사항"],
}

# 소분류: 항목 단위 키워드 → 세분화 라벨
ITEM_LEVEL_PATTERNS = {
    "budget":       ["사업예산", "사업금액", "총사업비", "추정가격", "추정금액",
                     "예산금액", "예산규모", "예정가격", "계약금액", "용역비",
                     "부가가치세", "VAT", "분할지급", "대가지급"],
    "period":       ["사업기간", "수행기간", "계약기간", "용역기간", "납품기한"],
    "warranty":     ["하자보수", "무상유지보수", "유지보수기간", "하자담보"],
    "project_name": ["사업명", "과업명", "용역명"],
    "deadline":     ["제출마감", "마감일시", "접수마감", "입찰마감", "마감일",
                     "제출기한", "입찰참여마감"],
    "budget_split": ["분할지급", "기성", "선급금", "잔금", "대가지급"],
    "eligibility":  ["참가자격", "입찰참가자격", "지원자격", "응찰자격"],
    "eval_method":  ["평가방법", "평가기준", "배점기준", "선정기준", "기술평가",
                     "가격평가", "정성평가"],
    "submission":   ["제출서류", "구비서류", "제안서규격", "제출방법"],
    "scope":        ["과업내용", "수행범위", "사업범위", "주요내용"],
    "contract":     ["계약방법", "계약체결", "입찰방식", "낙찰방법"],
    "penalty":      ["지체상금", "계약해지", "위약금", "손해배상"],
    "schedule":     ["추진일정", "수행일정", "납품일정", "마일스톤"],
}

HIGH_PRIORITY_SECTIONS = {
    "deadline", "submission_docs", "eligibility", "evaluation",
    "budget", "scope", "budget_split", "period", "eval_method",
    "submission", "penalty", "project_name",
}

@dataclass
class Section:
    section_title_raw: str
    section_label: str
    content: str
    start_line: int
    end_line: int
    is_high_priority: bool = False

@dataclass
class ItemChunk:
    """항목 단위 세분화 청크"""
    title: str           # "사업예산", "사업기간" 등
    label: str           # "budget", "period" 등
    parent_label: str    # 상위 섹션 라벨 ("overview" 등)
    content: str
    is_high_priority: bool = False


def normalize_for_body(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\u00A0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_for_label(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^[IVXivxⅠⅡⅢⅣⅤ①②③④⑤⑥⑦⑧⑨⑩]+[\.\s]*", "", text)
    text = re.sub(r"^[\-\•\·\▪\◦□■▶]+\s*", "", text)
    text = re.sub(r"^\(?[0-9가-힣ivxIVX]+\)?[.\)]\s*", "", text)
    text = re.sub(r"[\[\]{}()<>]", "", text)
    text = re.sub(r"[:：\-–—/]", "", text)
    text = re.sub(r"\s+", "", text)
    return text.strip()


def map_section_label(line: str) -> Optional[str]:
    target = normalize_for_label(line)
    for label, patterns in SECTION_PATTERNS.items():
        for p in patterns:
            if p in target:
                return label
    return None


def map_item_label(text: str) -> Optional[str]:
    """소분류 항목 매칭 (띄어쓰기 무시)"""
    target = re.sub(r"\s+", "", text)
    for label, patterns in ITEM_LEVEL_PATTERNS.items():
        for p in patterns:
            if p in target:
                return label
    return None


def is_heading_candidate(line: str) -> bool:
    raw = line.strip()
    if not raw or len(raw) > 60:
        return False
    if raw.endswith(("니다", "습니다", "음", "함", "됩니다")) and len(raw) > 15:
        return False
    heading_starters = re.compile(
        r"^([IVXivxⅠⅡⅢⅣⅤ①②③④⑤]|[0-9]+[.\)]|\([0-9가-힣]+\)|[가-힣A-Za-z])"
    )
    return bool(heading_starters.match(raw))


def is_item_heading(line: str) -> bool:
    """
    소항목 헤더 감지: "가.", "나.", "라.", "1)", "(1)", "□" 등으로 시작하고
    뒤에 키워드(사업명, 사업예산 등)가 오는 패턴
    """
    raw = line.strip()
    if not raw or len(raw) > 80:
        return False
    # "가. 사업명:", "라. 사업예산 :", "1) 사업기간" 등
    item_starter = re.compile(
        r"^([가-힣][.\)]\s*|[0-9]+[.\)]\s*|\([0-9]+\)\s*|□\s*|○\s*|●\s*|▶\s*)"
    )
    if item_starter.match(raw):
        return True
    # "사업예산 : 100,000,000원" 같은 직접 패턴
    direct_item = re.compile(
        r"^(사업[예금]산|사업기간|사업명|과업명|제출마감|마감일|참가자격|평가[기방]"
        r"|계약방법|무상유지보수|하자보수|제출서류|납품기한|수행기간)"
    )
    if direct_item.match(raw.replace(" ", "")):
        return True
    return False


def split_sections(text: str) -> List[Section]:
    """대분류 섹션 분리 (기존과 동일)"""
    text = normalize_for_body(text)
    lines = text.split("\n")
    heading_positions = []

    for idx, raw_line in enumerate(lines):
        if not raw_line.strip():
            continue
        if not is_heading_candidate(raw_line):
            continue
        label = map_section_label(raw_line)
        if label:
            heading_positions.append((idx, raw_line.strip(), label))

    if not heading_positions:
        return [Section("full_document", "full_document", text, 0, len(lines)-1)]

    sections = []
    for i, (start_idx, raw_title, label) in enumerate(heading_positions):
        end_idx = heading_positions[i+1][0] - 1 \
                  if i+1 < len(heading_positions) else len(lines)-1
        content = "\n".join(lines[start_idx+1:end_idx+1]).strip()
        sections.append(Section(
            section_title_raw=raw_title,
            section_label=label,
            content=content,
            start_line=start_idx,
            end_line=end_idx,
            is_high_priority=(label in HIGH_PRIORITY_SECTIONS),
        ))
    return sections


def split_section_into_items(section: Section) -> List[ItemChunk]:
    """
    ✅ 핵심: 대분류 섹션을 소항목 단위로 세분화
    "가. 사업명: ...\n나. 사업기간: ...\n다. 사업예산: ..."
    → [ItemChunk(title="사업명",...), ItemChunk(title="사업기간",...), ...]
    """
    lines = section.content.split("\n")
    items = []
    current_title = None
    current_label = None
    current_lines = []

    for line in lines:
        stripped = line.strip()
        if not stripped:
            current_lines.append(line)
            continue

        # 소항목 헤더인지 체크
        if is_item_heading(stripped):
            item_label = map_item_label(stripped)
            if item_label:
                # 이전 항목 저장
                if current_title and current_lines:
                    content = "\n".join(current_lines).strip()
                    if content:
                        items.append(ItemChunk(
                            title=current_title,
                            label=current_label or section.section_label,
                            parent_label=section.section_label,
                            content=content,
                            is_high_priority=(
                                (current_label or section.section_label) in HIGH_PRIORITY_SECTIONS
                            ),
                        ))
                current_title = stripped
                current_label = item_label
                current_lines = [line]
                continue

        current_lines.append(line)

    # 마지막 항목 저장
    if current_lines:
        content = "\n".join(current_lines).strip()
        if content:
            items.append(ItemChunk(
                title=current_title or section.section_title_raw,
                label=current_label or section.section_label,
                parent_label=section.section_label,
                content=content,
                is_high_priority=(
                    (current_label or section.section_label) in HIGH_PRIORITY_SECTIONS
                ),
            ))

    # 소항목을 못 찾은 경우 → 섹션 전체를 하나의 ItemChunk로
    if not items:
        items.append(ItemChunk(
            title=section.section_title_raw,
            label=section.section_label,
            parent_label=section.section_label,
            content=section.content,
            is_high_priority=section.is_high_priority,
        ))

    return items


def infer_chunk_type(text: str) -> str:
    if text.startswith("[표") or text.startswith("[추진일정표"):
        return "table"
    if text.startswith("[목차") or text.startswith("목차"):
        return "toc"
    bullet_like = len(re.findall(r"(^|\n)\s*[-•·▪◦가-힣0-9]+[.)]?\s+", text))
    if bullet_like >= 2:
        return "list"
    if len(text.split("\n")) <= 3 and len(text) < 100:
        return "short_text"
    return "text"


def split_long_text(
    text: str,
    max_chars: int = CHUNK_MAX_CHARS,
    overlap: int = CHUNK_OVERLAP
) -> List[str]:
    text = text.strip()
    if not text:
        return []
    if text.startswith("[표") or text.startswith("[추진일정표") or text.startswith("[목차"):
        return [text]
    if len(text) <= max_chars:
        return [text]

    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 2 <= max_chars:
            current = (current + "\n\n" + para).strip()
        else:
            if current:
                chunks.append(current)
            if len(para) > max_chars:
                start = 0
                while start < len(para):
                    end = min(len(para), start + max_chars)
                    chunks.append(para[start:end].strip())
                    if end == len(para):
                        break
                    start = max(0, end - overlap)
                current = ""
            else:
                current = para
    if current:
        chunks.append(current)
    return chunks


def build_chunks_from_sections(
    sections: List[Section],
    file_name: str,
    organization: Optional[str],
    project_name: Optional[str],
    doc_id: str,
) -> List[Dict[str, Any]]:
    """
    ✅ 개선: 섹션 → 소항목 세분화 → 청킹
    """
    output = []
    for sec in sections:
        # ✅ 핵심: 소항목 단위로 먼저 분리
        items = split_section_into_items(sec)

        for item in items:
            parts = split_long_text(item.content)
            for i, part in enumerate(parts, start=1):
                normalized = re.sub(r"\s+", "", part)
                output.append({
                    "text": part,
                    "text_normalized": normalized,
                    "metadata": {
                        "doc_id":                 doc_id,
                        "file_name":              file_name,
                        "organization":           organization,
                        "project_name":           project_name,
                        "section":                sec.section_title_raw,
                        "section_label":          item.label,       # ← 소분류 라벨
                        "parent_section_label":   item.parent_label, # ← 대분류 라벨
                        "item_title":             item.title,        # ← 항목 제목
                        "chunk_index_in_section": i,
                        "chunk_type":             infer_chunk_type(part),
                        "is_high_priority":       item.is_high_priority,
                    }
                })
    return output


# ── 테스트 ────────────────────────────────────────────
sample_text = """
Ⅰ. 사업개요

1. 사업개요
가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업
나. 사업기간: 계약일로부터 24개월 이내
다. 무상유지보수기간 : 사업종료일로부터 12개월
라. 사업예산 : 11,270,000,000원 (V.A.T 포함, 3년 분할 지급)
 - 2024학년도 약 30% 지급
 - 2025학년도 약 40% 지급
 - 2026학년도 약 30% 지급
마. 입찰 및 계약 방법: 제한 경쟁 입찰(협상에 의한 계약)
바. 본 사업은 3개년에 걸친 다년 사업으로, 사업 수행에 따른 대가도 3개년에 걸쳐 분할 지급됨

Ⅱ. 추진배경 및 필요성
기존 시스템 노후화로 인해 운영 효율이 저하되었다.

Ⅲ. 입찰참가자격
중소기업 이상

Ⅳ. 평가기준
정성평가 80점, 가격평가 20점
"""

test_sections = split_sections(sample_text)
print(f"대분류 섹션: {len(test_sections)}개\n")

all_items = []
for s in test_sections:
    items = split_section_into_items(s)
    all_items.extend(items)
    print(f"[{s.section_label}] {s.section_title_raw[:30]}")
    for item in items:
        priority = "⭐" if item.is_high_priority else "  "
        print(f"  {priority} → [{item.label}] {item.title[:40]} | {len(item.content)}자")

print(f"\n총 소항목: {len(all_items)}개")

# 청크 생성 테스트
test_chunks = build_chunks_from_sections(
    test_sections, "test.pdf", "고려대학교", "차세대 포털", "test-001"
)
print(f"총 chunk: {len(test_chunks)}개\n")
for c in test_chunks:
    m = c["metadata"]
    priority = "⭐" if m["is_high_priority"] else "  "
    print(f"  {priority} [{m['section_label']}] {m.get('item_title','')[:30]} | "
          f"{len(c['text'])}자 | {m['chunk_type']}")

대분류 섹션: 7개

[overview] Ⅰ. 사업개요
     → [overview] Ⅰ. 사업개요 | 0자
[overview] 1. 사업개요
  ⭐ → [project_name] 가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업 | 35자
  ⭐ → [period] 나. 사업기간: 계약일로부터 24개월 이내 | 23자
     → [warranty] 다. 무상유지보수기간 : 사업종료일로부터 12개월 | 27자
[budget] 라. 사업예산 : 11,270,000,000원 (V.A
  ⭐ → [budget] 라. 사업예산 : 11,270,000,000원 (V.A.T 포함, 3년  | 58자
[contract] 마. 입찰 및 계약 방법: 제한 경쟁 입찰(협상에 의한
  ⭐ → [budget] 바. 본 사업은 3개년에 걸친 다년 사업으로, 사업 수행에 따른 대가도  | 54자
[background] Ⅱ. 추진배경 및 필요성
     → [background] Ⅱ. 추진배경 및 필요성 | 28자
[eligibility] Ⅲ. 입찰참가자격
  ⭐ → [eligibility] Ⅲ. 입찰참가자격 | 7자
[evaluation] Ⅳ. 평가기준
  ⭐ → [evaluation] Ⅳ. 평가기준 | 18자

총 소항목: 9개
총 chunk: 8개

  ⭐ [project_name] 가. 사업명: 고려대학교 차세대 포털·학사 정보시스템  | 35자 | short_text
  ⭐ [period] 나. 사업기간: 계약일로부터 24개월 이내 | 23자 | short_text
     [warranty] 다. 무상유지보수기간 : 사업종료일로부터 12개월 | 27자 | short_text
  ⭐ [budget] 라. 사업예산 : 11,270,000,000원 (V.A | 58자 | list
  ⭐ [budget] 바. 본 사업은 3개년에 걸친 다년 사업으로, 사업 수 | 54자 | short_text
     [background] Ⅱ. 추진배경 및 필요성 | 28자 | s

In [152]:
# 셀 7: 문서 인덱스 + chunk 생성 — CSV 보완 + 신규 파일 추가 파이프라인
import uuid
import json
import pandas as pd
import unicodedata
from tqdm import tqdm

def normalize_fname(name: str) -> str:
    name = unicodedata.normalize("NFKC", name)
    name = name.strip()
    name = re.sub(r"[\s\u00A0\u3000]+", " ", name)
    return name

def build_metadata_lookup(df: pd.DataFrame) -> Dict[str, Dict]:
    """CSV 메타데이터를 정규화된 파일명으로 매핑"""
    lookup = {}
    col = "파일명" if "파일명" in df.columns else None
    if not col:
        return {}
    for _, row in df.iterrows():
        fname = str(row.get(col, "")).strip()
        if fname:
            norm = normalize_fname(fname)
            lookup[norm] = row.to_dict()
    return lookup


def estimate_text_quality(text: str) -> str:
    if len(text) < 300:
        return "low"
    weird = len(re.findall(r"[□■▣※◇◆△▽→←↑↓]", text)) / max(len(text), 1)
    return "medium" if weird > 0.02 else "high"


def process_single_document(path: Path, meta: Dict) -> tuple:
    file_name = path.name
    suffix = path.suffix.lower().replace(".", "")

    extracted = extract_text_from_file(str(path))
    csv_text = str(meta.get("텍스트", "")).strip()

    # ✅ 핵심 변경: HWP는 pyhwp2md(표) + CSV(본문) 결합
    if suffix in ["hwp", "hwpx"]:
        parts = []
        if csv_text:
            parts.append(csv_text)          # CSV 본문 (핵심)
        if extracted.strip():
            parts.append(extracted)          # pyhwp2md 표
        text = "\n\n".join(parts)
        text_source = "hwp_combined" if (csv_text and extracted.strip()) else \
                      "csv_fallback" if csv_text else \
                      "direct" if extracted.strip() else ""
    else:
        text = extracted.strip() if extracted.strip() else csv_text
        text_source = "direct" if extracted.strip() else "csv_fallback"

    if not text:
        return None, []

    organization = str(meta.get("발주 기관", "") or "").strip() or None
    project_name = str(meta.get("사업명", "") or "").strip() or None
    budget       = str(meta.get("사업 금액", "") or "").strip() or None
    bid_end_at   = str(meta.get("입찰 참여 마감일", "") or "").strip() or None
    published_at = str(meta.get("공개 일자", "") or "").strip() or None

    if not organization and "_" in Path(file_name).stem:
        organization = Path(file_name).stem.split("_")[0].strip()

    if not budget:
        m = re.search(
            r"(?:사업금액|예산금액|추정금액|사업예산|사업비)\s*[:：]?\s*([^\n]{3,30})",
            text
        )
        budget = m.group(1).strip() if m else None

    if not bid_end_at:
        m = re.search(
            r"(?:입찰참여마감일|제출마감|마감일)\s*[:：]?\s*([^\n]{5,30})",
            text
        )
        bid_end_at = m.group(1).strip() if m else None

    doc_id = str(uuid.uuid4())

    row = {
        "doc_id":       doc_id,
        "file_name":    file_name,
        "file_type":    suffix,
        "project_name": project_name,
        "organization": organization,
        "budget":       budget,
        "published_at": published_at,
        "bid_end_at":   bid_end_at,
        "text_length":  len(text),
        "text_source":  text_source,
        "has_table":    "|" in text or "[표" in text,
        "has_schedule": "[추진일정표" in text,
        "text_quality": estimate_text_quality(text),
    }

    sections = split_sections(text)
    chunks = build_chunks_from_sections(
        sections=sections,
        file_name=file_name,
        organization=organization,
        project_name=project_name,
        doc_id=doc_id,
    )
    return row, chunks


def process_all_documents(files_dir: Path, metadata_lookup: Dict) -> tuple:
    index_rows, all_chunks = [], []
    file_paths = sorted([
        p for p in files_dir.glob("*")
        if p.is_file() and p.suffix.lower() in [".pdf", ".hwp", ".hwpx", ".docx", ".docs"]
    ])
    print(f"처리 대상: {len(file_paths)}개 파일")

    for path in tqdm(file_paths, desc="문서 처리"):
        # ✅ 정규화된 파일명으로 메타 조회
        norm_name = normalize_fname(path.name)
        meta = metadata_lookup.get(norm_name, {})
        row, chunks = process_single_document(path, meta)
        if row:
            index_rows.append(row)
            all_chunks.extend(chunks)

    doc_index_df = pd.DataFrame(index_rows)
    print(f"\n완료: {len(doc_index_df)}개 문서, {len(all_chunks)}개 chunk")
    print(f"  표 포함 문서: {doc_index_df['has_table'].sum()}개")
    print(f"  일정표 포함: {doc_index_df['has_schedule'].sum()}개")
    print(f"  CSV fallback 사용: {(doc_index_df['text_source']=='csv_fallback').sum()}개")
    return doc_index_df, all_chunks


# ── 이슈 7: 신규 파일 추가 파이프라인 (고도화용) ─────────
def add_new_document(
    new_file_path: Path,
    meta: Dict,
    existing_chunks_path: Path,
    existing_index_path: Path,
):
    """
    새 RFP 파일 1개를 기존 DB에 추가.
    - document_index.json에 행 추가
    - chunks.json에 chunk 추가
    - embeddings는 별도로 재생성 필요 (임베딩 셀에서 처리)
    """
    row, new_chunks = process_single_document(new_file_path, meta)
    if not row:
        print(f"⚠️  텍스트 추출 실패: {new_file_path.name}")
        return

    # 기존 인덱스 로드 후 추가
    existing_df = pd.read_json(existing_index_path, orient="records")
    # 중복 방지
    if row["file_name"] in existing_df["file_name"].values:
        print(f"⚠️  이미 존재하는 파일: {row['file_name']} — 스킵")
        return

    updated_df = pd.concat([existing_df, pd.DataFrame([row])], ignore_index=True)
    updated_df.to_json(existing_index_path, orient="records", force_ascii=False, indent=2)

    # 기존 chunks 로드 후 추가
    with open(existing_chunks_path, "r", encoding="utf-8") as f:
        existing_chunks = json.load(f)
    updated_chunks = existing_chunks + new_chunks
    with open(existing_chunks_path, "w", encoding="utf-8") as f:
        json.dump(updated_chunks, f, ensure_ascii=False, indent=2)

    print(f"✓ 추가 완료: {row['file_name']} | chunk {len(new_chunks)}개")
    print(f"  총 문서: {len(updated_df)}개 / 총 chunk: {len(updated_chunks)}개")
    print(f"  ⚠️  임베딩 재생성 필요 — 셀 9 다시 실행하세요")

In [153]:
# ============================================================
# 셀 7-1: 테스트셋 10개 문서만 처리
# ============================================================
TEST_FILES = [
    "한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp",
    "인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp",
    "경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp",
    "고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf",
    "기초과학연구원_2025년도 중이온가속기용 극저온시스템 운전 용역.pdf",
    "서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf",
    "대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 및 시스템 고.hwp",
    "사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hwp",
    "을지대학교_을지대학교 비교과시스템 개발.hwp",
    "한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp",
]

# ✅ 정규화 매칭으로 파일 찾기
actual_files_map = {
    normalize_fname(f.name): f
    for f in FILES_DIR.glob("*") if f.is_file()
}

matched_files = []
missing_files = []

for test_name in TEST_FILES:
    norm_test = normalize_fname(test_name)
    if norm_test in actual_files_map:
        matched_files.append(actual_files_map[norm_test])
    else:
        missing_files.append(test_name)

print(f"매칭 성공: {len(matched_files)}개")
print(f"매칭 실패: {len(missing_files)}개")
if missing_files:
    for m in missing_files:
        print(f"  ⚠️ {m}")

print(f"\n=== 테스트셋 ===")
for f in matched_files:
    print(f"  {f.suffix.upper()[1:]:4s} | {f.name[:65]}")

매칭 성공: 10개
매칭 실패: 0개

=== 테스트셋 ===
  HWP  | 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 
  HWP  | 인천광역시_도시계획위원회 통합관리시스템 구축용역.hw
  HWP  | 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사어
  PDF  | 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
  PDF  | 기초과학연구원_2025년도 중이온가속기용 극저온시스템 우
  PDF  | 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 
  HWP  | 대한상공회의소_기업 재생에너지 지원센터 홈페이지 개편 미
  HWP  | 사단법인 보험개발원_실손보험 청구 전산화 시스템 구추
  HWP  | 을지대학교_을지대학교 비교과시스템 개발.hwp
  HWP  | 한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총


In [154]:
# ============================================================
# 셀 7-2: 테스트셋 10개 파싱 + 청킹
# ============================================================
import uuid, json
from tqdm import tqdm

meta_df = pd.read_csv(METADATA_CSV)
meta_df.columns = [c.strip() for c in meta_df.columns]
metadata_lookup = build_metadata_lookup(meta_df)

index_rows = []
all_chunks = []

for path in tqdm(matched_files, desc="테스트셋 처리"):
    # ✅ 정규화된 파일명으로 메타 조회
    norm_name = normalize_fname(path.name)
    meta = metadata_lookup.get(norm_name, {})
    row, chunks = process_single_document(path, meta)
    if row:
        index_rows.append(row)
        all_chunks.extend(chunks)
        print(f"  ✓ {path.name[:45]} | {row['text_source']:12s} | {len(chunks)} chunks")
    else:
        print(f"  ⚠️ {path.name[:45]} | 텍스트 없음")

doc_index_df = pd.DataFrame(index_rows)

print(f"\n{'='*60}")
print(f"총 문서: {len(doc_index_df)}개")
print(f"총 chunk: {len(all_chunks)}개")
print(f"문서당 평균: {len(all_chunks)/max(len(doc_index_df),1):.1f}개")
print(f"\n파싱 방법:")
print(doc_index_df["text_source"].value_counts())
print(f"\n파일형식:")
print(doc_index_df["file_type"].value_counts())

print(f"\n=== 문서별 chunk 수 ===")
from collections import Counter
doc_chunks = Counter(c["metadata"]["file_name"] for c in all_chunks)
for fname, count in doc_chunks.most_common():
    src = doc_index_df[doc_index_df["file_name"]==fname]["text_source"].values[0]
    print(f"  {count:4d} chunks | [{src:12s}] {fname[:55]}")

테스트셋 처리:  10%|█         | 1/10 [00:01<00:13,  1.50s/it]

  ✓ 한영대학_한영대학교 특성화 맞춤형  | hwp_combined | 15 chunks


테스트셋 처리:  20%|██        | 2/10 [00:04<00:18,  2.28s/it]

  ✓ 인천광역시_도시계획위원회 통합관리시 | hwp_combined | 14 chunks


테스트셋 처리:  30%|███       | 3/10 [00:06<00:14,  2.07s/it]

...

  ✓ 사단법인 보험개발원_실손보험 청구 ᄌ | hwp_combined | 25 chunks


테스트셋 처리:  90%|█████████ | 9/10 [02:43<00:13, 13.02s/it]

  ✓ 을지대학교_을지대학교 비교과시스템 개발 | hwp_combined | 19 chunks


테스트셋 처리: 100%|██████████| 10/10 [02:48<00:00, 16.83s/it]

  ✓ 한국철도공사 (용역)_모바일오피스 시스ᄐ | hwp_combined | 136 chunks

총 문서: 10개
총 chunk: 1213개
문서당 평균: 121.3개

파싱 방법:
text_source
hwp_combined    7
direct          3
Name: count, dtype: int64

파일형식:
file_type
hwp    7
pdf    3
Name: count, dtype: int64

=== 문서별 chunk 수 ===
   799 chunks | [direct      ] 고려대학교_차세대 포털·학사 정보시스템 구축사업.
   162 chunks | [direct      ] 서울특별시_2024년 지도정보 플랫폼 및 전문활
   136 chunks | [hwp_combined] 한국철도공사 (용역)_모바일오피스 시스템 고도화 
    25 chunks | [hwp_combined] 사단법인 보험개발원_실손보험 청구 전산화 시
    21 chunks | [hwp_combined] 경상북도 봉화군_봉화군 재난통합관리시스템 ᄀ
    19 chunks | [hwp_combined] 을지대학교_을지대학교 비교과시스템 개발.hwp
    15 chunks | [hwp_combined] 한영대학_한영대학교 특성화 맞춤형 교육환겨
    14 chunks | [hwp_combined] 인천광역시_도시계획위원회 통합관리시스템 구추
    13 chunks | [direct    

In [155]:
# 셀 8 수정본: 파싱 결과 JSON 저장
import json
from pathlib import Path

def save_artifacts(doc_index_df, chunks):
    ARTIFACTS.mkdir(parents=True, exist_ok=True)

    # 1) 문서 인덱스 → JSON 저장
    doc_index_json = ARTIFACTS / "document_index.json"
    doc_index_df.to_json(
        doc_index_json,
        orient="records",
        force_ascii=False,
        indent=2
    )
    print(f"✓ {doc_index_json} ({len(doc_index_df)}개 문서)")

    # 2) chunks → JSON 저장 (기존과 동일)
    chunks_json = ARTIFACTS / "chunks.json"
    with open(chunks_json, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    print(f"✓ {chunks_json} ({len(chunks)}개 chunk)")

    # 3) 저장 확인
    print(f"\n[저장 확인]")
    for p in [doc_index_json, chunks_json]:
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name}: {size_kb:.1f} KB")

save_artifacts(doc_index_df, all_chunks)

✓ /home/spai0703/workspace/artifacts/document_index.json (10개 문서)
✓ /home/spai0703/workspace/artifacts/chunks.json (1213개 chunk)

[저장 확인]
  document_index.json: 6.7 KB
  chunks.json: 3244.0 KB


In [156]:
# 셀 로드: JSON에서 불러오기
import json
import pandas as pd

def load_artifacts():
    doc_index_df = pd.read_json(
        ARTIFACTS / "document_index.json",
        orient="records"
    )
    with open(ARTIFACTS / "chunks.json", "r", encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"✓ 문서 인덱스: {len(doc_index_df)}개")
    print(f"✓ chunks: {len(chunks)}개")
    return doc_index_df, chunks

# 사용
doc_index_df, all_chunks = load_artifacts()

✓ 문서 인덱스: 10개
✓ chunks: 1213개


In [157]:
# ============================================================
# 셀 9: Chroma + CSV + 쿼리확장 + 메타필터 + Hybrid Retriever
# ============================================================

from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from typing import List, Optional, Dict
from pydantic import Field
from tqdm import tqdm
import pandas as pd
import json
import re
import os

# ═══════════════════════════════════════════════════════
# 1. 쿼리 확장 (한국어 RFP 표현 다양성 대응)
# ═══════════════════════════════════════════════════════

QUERY_EXPANSION_MAP = {
    "예산":   ["사업예산", "총사업비", "예산규모", "사업금액", "추정가격",
               "추정금액", "예정가격", "용역비", "계약금액", "예산액"],
    "마감":   ["제출마감", "마감일", "접수마감", "입찰마감", "마감일시",
               "제출기한", "참여마감"],
    "자격":   ["참가자격", "입찰참가자격", "지원자격", "응찰자격",
               "입찰자격요건"],
    "평가":   ["평가기준", "평가방법", "배점기준", "선정기준", "기술평가",
               "가격평가", "정성평가", "정량평가"],
    "기간":   ["사업기간", "수행기간", "계약기간", "용역기간", "납품기한"],
    "서류":   ["제출서류", "구비서류", "제안서", "입찰등록서류"],
    "범위":   ["과업내용", "사업내용", "수행범위", "과업범위", "사업범위"],
    "계약":   ["계약방법", "계약조건", "낙찰방법", "입찰방식"],
    "일정":   ["추진일정", "수행일정", "마일스톤", "납품일정"],
    "벌칙":   ["지체상금", "위약금", "손해배상", "계약해지"],
    "유지보수": ["하자보수", "무상유지보수", "유지보수기간", "하자담보"],
}

# 쿼리 키워드 → 관련 section_label 매핑
QUERY_TO_SECTION_FILTER = {
    "예산":   ["budget", "budget_split"],
    "마감":   ["deadline"],
    "자격":   ["eligibility"],
    "평가":   ["eval_method", "evaluation"],
    "기간":   ["period"],
    "서류":   ["submission", "submission_docs"],
    "범위":   ["scope"],
    "계약":   ["contract"],
    "일정":   ["schedule"],
    "유지보수": ["warranty"],
}


def expand_query(query: str) -> str:
    """쿼리에 동의어/유사 표현 추가"""
    expanded_terms = []
    q_norm = re.sub(r"\s+", "", query)

    for keyword, synonyms in QUERY_EXPANSION_MAP.items():
        if keyword in q_norm:
            expanded_terms.extend(synonyms[:4])  # 상위 4개만

    if expanded_terms:
        return query + " " + " ".join(set(expanded_terms))
    return query


def detect_section_filter(query: str) -> Optional[List[str]]:
    """쿼리에서 관련 section_label 필터 추출"""
    q_norm = re.sub(r"\s+", "", query)
    labels = []
    for keyword, section_labels in QUERY_TO_SECTION_FILTER.items():
        if keyword in q_norm:
            labels.extend(section_labels)
    return labels if labels else None


# ═══════════════════════════════════════════════════════
# 2. CSV 메타 Retriever
# ═══════════════════════════════════════════════════════

class CSVMetaRetriever(BaseRetriever):
    """
    data_list.csv의 구조화 정보를 검색.
    - 컬럼명 공백 자동 처리
    - 검색 시 띄어쓰기 무시
    - 사업요약 텍스트도 검색 대상에 포함
    """
    docs: List[Document] = Field(default_factory=list)
    k: int = 3

    @classmethod
    def from_csv(cls, csv_path: str, k: int = 3) -> "CSVMetaRetriever":
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]

        # 컬럼명 매핑 (공백 있는 원본 → 정규화)
        col_map = {}
        for c in df.columns:
            col_map[re.sub(r"\s+", "", c)] = c
        
        def get_col(key_no_space: str) -> str:
            """띄어쓰기 무시하고 컬럼 찾기"""
            return col_map.get(key_no_space, "")

        docs = []
        for _, row in df.iterrows():
            def safe(col_key: str) -> str:
                col_name = get_col(col_key)
                if not col_name:
                    return ""
                val = row.get(col_name, "")
                return str(val).strip() if pd.notna(val) else ""

            file_name   = safe("파일명")
            org         = safe("발주기관")
            proj        = safe("사업명")
            budget      = safe("사업금액")
            bid_start   = safe("입찰참여시작일")
            bid_end     = safe("입찰참여마감일")
            published   = safe("공개일자")
            summary     = safe("사업요약")
            ann_number  = safe("공고번호")
            ann_round   = safe("공고차수")

            # 구조화 텍스트 생성 (검색 + LLM 근거용)
            parts = []
            if proj:       parts.append(f"사업명: {proj}")
            if org:        parts.append(f"발주기관: {org}")
            if budget:     parts.append(f"사업예산: {budget}")
            if ann_number: parts.append(f"공고번호: {ann_number}")
            if ann_round:  parts.append(f"공고차수: {ann_round}")
            if published:  parts.append(f"공개일자: {published}")
            if bid_start:  parts.append(f"입찰시작일: {bid_start}")
            if bid_end:    parts.append(f"입찰마감일: {bid_end}")
            if summary:    parts.append(f"사업요약: {summary}")

            content = "\n".join(parts)
            if not content.strip():
                continue

            docs.append(Document(
                page_content=content,
                metadata={
                    "file_name":      file_name,
                    "organization":   org,
                    "project_name":   proj,
                    "budget":         budget,
                    "bid_start_at":   bid_start,
                    "bid_end_at":     bid_end,
                    "published_at":   published,
                    "ann_number":     ann_number,
                    "section_label":  "csv_meta",
                    "item_title":     "CSV 메타정보",
                    "chunk_type":     "csv_meta",
                    "is_high_priority": True,
                    "source":         "data_list.csv",
                }
            ))

        print(f"  CSV 메타 Document: {len(docs)}개 생성")
        print(f"  CSV 컬럼 매핑: {list(col_map.keys())}")
        return cls(docs=docs, k=k)

    def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        q_norm = re.sub(r"\s+", "", query.lower())
        scored = []

        for doc in self.docs:
            # 전체 content + 메타 합쳐서 검색
            content_norm = re.sub(r"\s+", "", doc.page_content.lower())
            meta_text = (
                doc.metadata.get("project_name", "") +
                doc.metadata.get("organization", "") +
                doc.metadata.get("budget", "") +
                doc.metadata.get("ann_number", "")
            )
            meta_norm = re.sub(r"\s+", "", meta_text.lower())

            score = 0
            # 전체 쿼리 (띄어쓰기 제거) 매칭
            if q_norm in meta_norm:
                score += 10
            if q_norm in content_norm:
                score += 5

            # 개별 단어 매칭
            for word in query.split():
                w_norm = re.sub(r"\s+", "", word.lower())
                if len(w_norm) < 2:
                    continue
                if w_norm in meta_norm:
                    score += 5
                if w_norm in content_norm:
                    score += 2

            # 쿼리 확장 키워드 매칭
            expanded = expand_query(query)
            for exp_word in expanded.split():
                exp_norm = re.sub(r"\s+", "", exp_word.lower())
                if len(exp_norm) < 2:
                    continue
                if exp_norm in content_norm:
                    score += 1

            if score > 0:
                scored.append((score, doc))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [doc for _, doc in scored[:self.k]]


# ═══════════════════════════════════════════════════════
# 3. 정규화 BM25 Retriever (띄어쓰기 무시 + 쿼리 확장)
# ═══════════════════════════════════════════════════════

class NormalizedBM25Retriever(BaseRetriever):
    base_retriever: BM25Retriever = Field(default=None)
    original_docs: List[Document] = Field(default_factory=list)
    k: int = 5

    @classmethod
    def from_documents(cls, docs: List[Document], k: int = 5) -> "NormalizedBM25Retriever":
        normalized_docs = []
        for doc in docs:
            norm_text = re.sub(r"\s+", "", doc.page_content)
            normalized_docs.append(Document(
                page_content=norm_text,
                metadata=doc.metadata
            ))
        bm25 = BM25Retriever.from_documents(normalized_docs)
        bm25.k = k
        return cls(base_retriever=bm25, original_docs=docs, k=k)

    def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        # ✅ 쿼리 확장 적용
        expanded = expand_query(query)
        norm_query = re.sub(r"\s+", "", expanded)
        norm_results = self.base_retriever.invoke(norm_query)

        result_docs = []
        for nr in norm_results:
            norm_content = nr.page_content
            for orig in self.original_docs:
                if re.sub(r"\s+", "", orig.page_content) == norm_content:
                    result_docs.append(orig)
                    break
        return result_docs[:self.k]


# ═══════════════════════════════════════════════════════
# 4. Ensemble Retriever (RRF + 메타필터 + Rerank)
# ═══════════════════════════════════════════════════════

class SmartEnsembleRetriever(BaseRetriever):
    """
    BM25 + Vector + CSV를 RRF로 결합
    + 메타데이터 필터 부스트
    + 고빈도 섹션 부스트
    + 쿼리 기반 section_label 부스트
    """
    retrievers: list = Field(default_factory=list)
    weights: List[float] = Field(default_factory=lambda: [0.3, 0.5, 0.2])
    c: int = 60
    top_k: int = 8  # 최종 반환 수 (여유있게)

    def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        # 쿼리에서 관련 섹션 라벨 추출
        target_labels = detect_section_filter(query)

        all_docs = []
        for retriever in self.retrievers:
            try:
                docs = retriever.invoke(query)
                all_docs.append(docs)
            except Exception as e:
                print(f"  [Retriever 오류] {e}")
                all_docs.append([])

        doc_scores = {}
        for i, docs in enumerate(all_docs):
            weight = self.weights[i] if i < len(self.weights) else 0.1
            for rank, doc in enumerate(docs):
                key = doc.page_content
                if key not in doc_scores:
                    doc_scores[key] = [0.0, doc]

                base_score = weight * (1.0 / (rank + self.c))
                doc_scores[key][0] += base_score

        # ── Rerank 부스트 ────────────────────────────
        for key, (score, doc) in doc_scores.items():
            md = doc.metadata
            boost = 1.0

            # (1) 고빈도 섹션 부스트
            if md.get("is_high_priority"):
                boost *= 1.3

            # (2) 쿼리 관련 section_label 부스트
            if target_labels:
                doc_label = md.get("section_label", "")
                if doc_label in target_labels:
                    boost *= 2.0  # 핵심 매칭
                parent_label = md.get("parent_section_label", "")
                if parent_label in target_labels:
                    boost *= 1.5

            # (3) CSV 메타는 항상 약간 부스트
            if md.get("source") == "data_list.csv":
                boost *= 1.2

            # (4) 쿼리 키워드가 본문에 직접 등장하면 부스트
            q_norm = re.sub(r"\s+", "", query.lower())
            content_norm = re.sub(r"\s+", "", doc.page_content.lower())
            for kw in [q_norm] + [re.sub(r"\s+", "", w) for w in query.split()]:
                if len(kw) >= 2 and kw in content_norm:
                    boost *= 1.1

            doc_scores[key][0] = score * boost

        sorted_docs = sorted(doc_scores.values(), key=lambda x: x[0], reverse=True)
        return [doc for _, doc in sorted_docs[:self.top_k]]


# ═══════════════════════════════════════════════════════
# 5. 빌드 실행
# ═══════════════════════════════════════════════════════

CHROMA_DIR = str(ARTIFACTS / "chroma_db")

embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

def chunks_to_langchain_docs(chunks: list) -> list:
    docs = []
    for c in chunks:
        docs.append(Document(
            page_content=c["text"],
            metadata={k: (v if v is not None else "") for k, v in c["metadata"].items()}
        ))
    return docs

lc_docs = chunks_to_langchain_docs(all_chunks)
print(f"LangChain Document 변환: {len(lc_docs)}개")

# ── Chroma 저장 ──────────────────────────────────────
BATCH_SIZE = 100
TEST_MODE = False   # ⚠️ 전체 실행 시 False

target_docs = lc_docs[:30] if TEST_MODE else lc_docs  # ← 10→30 (세분화됐으므로)
print(f"임베딩 대상: {len(target_docs)}개 {'(테스트)' if TEST_MODE else '(전체)'}")

# 기존 컬렉션 초기화 (중복 방지)
vectorstore = Chroma(
    collection_name="rfp_chunks_v2",   # ← 버전업
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

# 기존 데이터 삭제 후 재저장
try:
    existing = vectorstore._collection.count()
    if existing > 0:
        print(f"  기존 {existing}개 삭제 중...")
        vectorstore._collection.delete(where={"doc_id": {"$ne": ""}})
except:
    pass

for i in tqdm(range(0, len(target_docs), BATCH_SIZE), desc="Chroma 저장"):
    batch = target_docs[i:i+BATCH_SIZE]
    vectorstore.add_documents(batch)

print(f"✓ Chroma DB 저장 완료: {CHROMA_DIR}")
print(f"  총 저장 문서 수: {vectorstore._collection.count()}")

# ── 3개 Retriever 구성 ───────────────────────────────

# (1) Vector Retriever
vector_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": RETRIEVAL_TOP_K, "fetch_k": RETRIEVAL_TOP_K * 3},
)

# (2) 정규화 BM25 (띄어쓰기 무시 + 쿼리 확장)
bm25_retriever = NormalizedBM25Retriever.from_documents(target_docs, k=RETRIEVAL_TOP_K)

# (3) CSV 메타 Retriever
csv_retriever = CSVMetaRetriever.from_csv(str(METADATA_CSV), k=3)

# Hybrid: BM25 30% + Vector 50% + CSV 20%
hybrid_retriever = SmartEnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever, csv_retriever],
    weights=[0.3, 0.5, 0.2],
    top_k=8,
)

print("✓ Smart Hybrid Retriever 준비 완료")
print("  BM25 30% (정규화+쿼리확장) + Vector 50% (MMR) + CSV 20%")
print("  + section_label 필터 부스트 + 고빈도 섹션 부스트")

LangChain Document 변환: 1213개
임베딩 대상: 1213개 (전체)
  기존 1213개 삭제 중...


Chroma 저장: 100%|██████████| 13/13 [00:20<00:00,  1.59s/it]

✓ Chroma DB 저장 완료: /home/spai0703/workspace/artifacts/chroma_db
  총 저장 문서 수: 1213
  CSV 메타 Document: 100개 생성
  CSV 컬럼 매핑: ['공고번호', '공고차수', '사업명', '사업금액', '발주기관', '공개일자', '입찰참여시작일', '입찰참여마감일', '사업요약', '파일형식', '파일명', '텍스트']
✓ Smart Hybrid Retriever 준비 완료
  BM25 30% (정규화+쿼리확장) + Vector 50% (MMR) + CSV 20%
  + section_label 필터 부스트 + 고빈도 섹션 부스트


In [158]:
# ============================================================
# 셀 10: RAG 체인 — 페르소나 + 마감 D-day + 답변 생성
# ============================================================
from datetime import date
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import re

TODAY = date.today()

# ── 페르소나 + 프롬프트 ──────────────────────────────
SYSTEM_PROMPT = f"""당신은 공공입찰 컨설팅 스타트업 '입찰메이트'의 RFP 분석 전문가입니다.
오늘 날짜는 {TODAY} 입니다.

[역할]
- 컨설턴트가 RFP 문서를 빠르게 파악할 수 있도록 핵심 정보를 정확하게 전달합니다.
- 입찰 마감일이 오늘 이후인지 반드시 확인하고 D-day를 알려주세요.

[답변 원칙]
1. 반드시 제공된 문서 근거만 사용하세요.
2. 근거에 없는 내용은 절대 추측하지 말고 "문서에서 확인되지 않음"이라고 답하세요.
3. 답변 끝에 반드시 사용한 파일명과 섹션을 "📎 근거:" 형태로 정리하세요.
4. 마감일 정보가 있으면 오늘({TODAY}) 기준 D-day를 계산하세요.
5. 핵심 정보는 bullet 형식으로 간결하게 정리하세요.
6. CSV 메타 정보(csv_meta)도 근거로 활용 가능합니다.
"""

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """다음 문서 근거를 바탕으로 질문에 답변하세요.

[문서 근거]
{context}

[질문]
{question}"""),
])

# ── LLM ──────────────────────────────────────────────
llm = ChatOpenAI(
    model=GEN_MODEL,
    temperature=0.1,
    max_tokens=1500,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)


def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        md = doc.metadata
        priority_mark = "⭐" if md.get("is_high_priority") else ""
        source_mark = "[CSV]" if md.get("source") == "data_list.csv" else "[문서]"

        formatted.append(
            f"[근거 {i}] {priority_mark} {source_mark}\n"
            f"- 파일: {md.get('file_name', '?')}\n"
            f"- 기관: {md.get('organization', '?')}\n"
            f"- 섹션: {md.get('section_label', '?')}\n"
            f"- 내용:\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


def calculate_dday(text: str) -> str:
    date_pattern = re.compile(r"(\d{4})[.\-/년](\d{1,2})[.\-/월](\d{1,2})")
    matches = date_pattern.findall(text)
    if not matches:
        return ""
    results = []
    for y, m, d in matches:
        try:
            target = date(int(y), int(m), int(d))
            diff = (target - TODAY).days
            if diff >= 0:
                results.append(f"📅 마감 D-{diff} ({target})")
            else:
                results.append(f"📅 마감 {abs(diff)}일 경과 ({target})")
        except ValueError:
            continue
    return "\n".join(results)


# ── RAG 체인 ─────────────────────────────────────────
rag_chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)


def answer_question(question: str) -> str:
    print(f"\n{'='*60}")
    print(f"질문: {question}")
    print(f"{'='*60}")

    retrieved = hybrid_retriever.invoke(question)
    print(f"\n검색된 chunk: {len(retrieved)}개")
    for i, doc in enumerate(retrieved, 1):
        md = doc.metadata
        priority = "⭐" if md.get("is_high_priority") else ""
        source = "[CSV]" if md.get("source") == "data_list.csv" else "[문서]"
        print(f"  {i}. {priority} {source} {md.get('file_name','?')[:35]} | "
              f"{md.get('section_label','?')}")

    context_text = " ".join(doc.page_content for doc in retrieved)
    dday = calculate_dday(context_text)
    if dday:
        print(f"\n{dday}")

    answer = rag_chain.invoke(question)
    print(f"\n[답변]\n{answer}")
    return answer


# ── 테스트 ───────────────────────────────────────────
test_questions = [
    "을지대학교 비교과시스템 개발 사업 예산은?",
    "한국철도공사 모바일오피스 입찰 참가자격은?",
    "서울특별시 지도정보 플랫폼 사업의 평가기준을 알려줘",
    "보험개발원 실손보험 시스템 제출서류는?",
    "인천광역시 도시계획위원회 사업기간은?",
]

for q in test_questions:
    answer_question(q)


질문: 을지대학교 비교과시스템 개발 사업 예산은?

검색된 chunk: 8개
  1. ⭐ [문서] 을지대학교_을지대학교 비교과시ᄉ | project_name
  2. ⭐ [문서] 을지대학교_을지대학교 비교과시ᄉ | scope
  3. ⭐ [문서] 고려대학교_차세대 포털·학사 정ᄇ | evaluation
  4. ⭐ [문서] 고려대학교_차세대 포털·학사 정ᄇ | scope
  5.  [문서] 고려대학교_차세대 포털·학사 정ᄇ | background
  6. ⭐ [CSV] 을지대학교_을지대학교 비교과시스템 개발.hwp | csv_meta
  7. ⭐ [CSV] 국민연금공단_사업장 사회보험료 지원 고시 개정에 따른 정보시스템 | csv_meta
  8. ⭐ [CSV] 사단법인 보험개발원_실손보험 청구 전산화 시스템 구축 사업.hw | csv_meta

📅 마감 497일 경과 (2024-11-27)
📅 마감 489일 경과 (2024-12-05)
📅 마감 672일 경과 (2024-06-05)
📅 마감 649일 경과 (2024-06-28)
📅 마감 645일 경과 (2024-07-02)
📅 마감 831일 경과 (2023-12-29)
📅 마감 750일 경과 (2024-03-19)
📅 마감 742일 경과 (2024-03-27)

[답변]
- 을지대학교 비교과시스템 개발 사업의 예산은 0.0입니다.

📎 근거: 을지대학교_을지대학교 비교과시스템 개발.hwp, csv_meta

질문: 한국철도공사 모바일오피스 입찰 참가자격은?

검색된 chunk: 8개
  1. ⭐ [문서] 서울특별시_2024년 지도정보 플 | budget
  2.  [문서] 고려대학교_차세대 포털·학사 정ᄇ | submission_method
  3.  [문서] 한국철도공사 (ᄋ

In [159]:
# ============================================================
# 셀 11: 평가셋(Gold Set) 구축
# ============================================================
# 4유형 질문셋 + Ground Truth + 근거(Evidence) 포함
 
import json
import pandas as pd
from datetime import datetime
 
EVAL_SET_PATH = ARTIFACTS / "eval_set.json"
 
# ── 평가 질문 유형 ────────────────────────────────────
# Type 1: 단일 문서 정보 추출 (single_doc)
# Type 2: 다문서 비교/종합 (multi_doc)
# Type 3: 멀티턴 후속질문 (multi_turn) — 향후 확장
# Type 4: 무응답/거절 (unanswerable)
 
eval_questions = [
    # ── Type 1: 단일 문서 정보 추출 ─────────────────
    {
        "question_id": "q01",
        "question": "고려대학교 차세대 포털 사업의 예산은 얼마인가요?",
        "question_type": "single_doc",
        "target_doc_keywords": ["고려대학교", "차세대 포털"],
        "gold_answer": "11,270,000,000원 (V.A.T 포함, 3년 분할 지급)",
        "gold_evidence": "라. 사업예산 : 11,270,000,000원 (V.A.T 포함, 3년 분할 지급)",
        "gold_section": "budget",
        "is_answerable": True,
    },
    {
        "question_id": "q02",
        "question": "봉화군 재난통합관리시스템 고도화 사업 예산을 알려줘",
        "question_type": "single_doc",
        "target_doc_keywords": ["봉화군", "재난통합관리"],
        "gold_answer": "900,000,000원",
        "gold_evidence": "사업금액 900,000,000원",
        "gold_section": "budget",
        "is_answerable": True,
    },
    {
        "question_id": "q03",
        "question": "서울특별시 지도정보 플랫폼 사업의 평가기준을 알려줘",
        "question_type": "single_doc",
        "target_doc_keywords": ["서울특별시", "지도정보"],
        "gold_answer": "제안서 평가(기술능력평가 90점 + 입찰가격 평가 10점)와 평가항목(품질보증, 시험운영, 교육훈련, 유지보수, 기밀보안, 비상대책, 하도급 계획 등)",
        "gold_evidence": "기술능력 평가: 90점, 입찰가격 평가: 10점",
        "gold_section": "evaluation",
        "is_answerable": True,
    },
    {
        "question_id": "q04",
        "question": "한국철도공사 모바일오피스 시스템 고도화 용역의 입찰 참가자격은?",
        "question_type": "single_doc",
        "target_doc_keywords": ["한국철도공사", "모바일오피스"],
        "gold_answer": "국가를 당사자로 하는 계약에 관한 법률 시행령에 따른 경쟁입찰 참가자격을 갖춘 소프트웨어사업자, 중소기업자만 입찰참가 가능",
        "gold_evidence": "입찰 참가자격: 소프트웨어 진흥법에 따라 소프트웨어사업자로 신고된 업체, 중소기업자만 입찰참가 가능",
        "gold_section": "eligibility",
        "is_answerable": True,
    },
    {
        "question_id": "q05",
        "question": "인천광역시 도시계획위원회 사업기간은?",
        "question_type": "single_doc",
        "target_doc_keywords": ["인천광역시", "도시계획위원회"],
        "gold_answer": "착수일로부터 180일",
        "gold_evidence": "사업 기간은 착수일로부터 180일",
        "gold_section": "period",
        "is_answerable": True,
    },
    {
        "question_id": "q06",
        "question": "을지대학교 비교과시스템 개발 사업 예산은?",
        "question_type": "single_doc",
        "target_doc_keywords": ["을지대학교", "비교과"],
        "gold_answer": "CSV 메타데이터에서 확인 필요 (문서 본문에서 명시적 예산 미확인)",
        "gold_evidence": "",
        "gold_section": "budget",
        "is_answerable": True,
    },
    {
        "question_id": "q07",
        "question": "보험개발원 실손보험 청구 전산화 시스템의 제출서류는 무엇인가요?",
        "question_type": "single_doc",
        "target_doc_keywords": ["보험개발원", "실손보험"],
        "gold_answer": "RFP 문서 내 제출서류 섹션 참조",
        "gold_evidence": "",
        "gold_section": "submission_docs",
        "is_answerable": True,
    },
    {
        "question_id": "q08",
        "question": "대한상공회의소 재생에너지 지원센터 홈페이지 사업의 사업기간은?",
        "question_type": "single_doc",
        "target_doc_keywords": ["대한상공회의소", "재생에너지"],
        "gold_answer": "RFP 문서 내 사업기간 참조",
        "gold_evidence": "",
        "gold_section": "period",
        "is_answerable": True,
    },
 
    # ── Type 2: 다문서 비교/종합 ─────────────────────
    {
        "question_id": "q09",
        "question": "테스트 문서 중 사업 예산이 가장 큰 사업은 어디인가요?",
        "question_type": "multi_doc",
        "target_doc_keywords": [],
        "gold_answer": "고려대학교 차세대 포털·학사 정보시스템 구축사업 (11,270,000,000원)",
        "gold_evidence": "CSV 및 문서 예산 정보 비교",
        "gold_section": "budget",
        "is_answerable": True,
    },
    {
        "question_id": "q10",
        "question": "교육 관련 시스템 구축 사업을 모두 알려줘",
        "question_type": "multi_doc",
        "target_doc_keywords": ["교육", "학사", "비교과", "포털"],
        "gold_answer": "고려대학교 차세대 포털·학사 정보시스템, 을지대학교 비교과시스템, 한영대학교 특성화 맞춤형 교육환경 구축 등",
        "gold_evidence": "각 문서 사업명에서 교육 관련 키워드 확인",
        "gold_section": "overview",
        "is_answerable": True,
    },
 
    # ── Type 4: 무응답/거절 ──────────────────────────
    {
        "question_id": "q11",
        "question": "고려대학교 차세대 포털 사업 담당자의 휴대폰 번호를 알려줘",
        "question_type": "unanswerable",
        "target_doc_keywords": ["고려대학교"],
        "gold_answer": "문서에서 확인되지 않음",
        "gold_evidence": "",
        "gold_section": "",
        "is_answerable": False,
    },
    {
        "question_id": "q12",
        "question": "봉화군 사업에서 사용할 클라우드 서비스 제공업체는 어디인가요?",
        "question_type": "unanswerable",
        "target_doc_keywords": ["봉화군"],
        "gold_answer": "문서에서 확인되지 않음",
        "gold_evidence": "",
        "gold_section": "",
        "is_answerable": False,
    },
]
 
# 저장
with open(EVAL_SET_PATH, "w", encoding="utf-8") as f:
    json.dump(eval_questions, f, ensure_ascii=False, indent=2)
 
print(f"✓ 평가셋 저장 완료: {EVAL_SET_PATH}")
print(f"  총 {len(eval_questions)}개 질문")
print(f"  유형별: single_doc {sum(1 for q in eval_questions if q['question_type']=='single_doc')}개"
      f" / multi_doc {sum(1 for q in eval_questions if q['question_type']=='multi_doc')}개"
      f" / unanswerable {sum(1 for q in eval_questions if q['question_type']=='unanswerable')}개")

✓ 평가셋 저장 완료: /home/spai0703/workspace/artifacts/eval_set.json
  총 12개 질문
  유형별: single_doc 8개 / multi_doc 2개 / unanswerable 2개


In [160]:
# ============================================================
# 셀 12: 배치 실행 + 실행 로그 저장
# ============================================================
import time
 
PREDICTIONS_PATH = ARTIFACTS / "predictions.json"
 
def run_eval_batch(questions: list, retriever, rag_chain) -> list:
    """
    평가셋 전체를 RAG에 돌리고 실행 로그를 저장.
    retrieval 결과 + 생성 답변 + 메타데이터 모두 기록.
    """
    predictions = []
 
    for q in questions:
        qid = q["question_id"]
        question = q["question"]
        print(f"\n[{qid}] {question}")
 
        start_time = time.time()
 
        # Retrieval
        try:
            retrieved_docs = retriever.invoke(question)
        except Exception as e:
            print(f"  ❌ Retrieval 오류: {e}")
            retrieved_docs = []
 
        # 검색 결과 메타 수집
        retrieved_info = []
        for i, doc in enumerate(retrieved_docs):
            md = doc.metadata
            retrieved_info.append({
                "rank": i + 1,
                "file_name": md.get("file_name", ""),
                "section_label": md.get("section_label", ""),
                "item_title": md.get("item_title", ""),
                "is_high_priority": md.get("is_high_priority", False),
                "source": md.get("source", "document"),
                "content_preview": doc.page_content[:200],
            })
 
        # Generation
        try:
            answer = rag_chain.invoke(question)
        except Exception as e:
            print(f"  ❌ Generation 오류: {e}")
            answer = f"[오류] {e}"
 
        latency = time.time() - start_time
 
        prediction = {
            "question_id": qid,
            "question": question,
            "question_type": q["question_type"],
            "is_answerable": q["is_answerable"],
            "gold_answer": q["gold_answer"],
            "gold_section": q.get("gold_section", ""),
            "generated_answer": answer,
            "retrieved_docs": retrieved_info,
            "num_retrieved": len(retrieved_docs),
            "latency_sec": round(latency, 2),
            "timestamp": datetime.now().isoformat(),
        }
        predictions.append(prediction)
 
        # 간략 출력
        status = "✓" if answer and len(answer) > 10 else "⚠️"
        print(f"  {status} 답변: {answer[:100]}...")
        print(f"  검색: {len(retrieved_docs)}개 | {latency:.1f}초")
 
    # 저장
    with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    print(f"\n✓ 예측 결과 저장: {PREDICTIONS_PATH} ({len(predictions)}개)")
 
    return predictions
 
 
# 실행
predictions = run_eval_batch(eval_questions, hybrid_retriever, rag_chain)


[q01] 고려대학교 차세대 포털 사업의 예산은 얼마인가요?
  ✓ 답변: - 고려대학교 차세대 포털·학사 정보시스템 구축 사업의 예산은 11,270,000,000원입니다.

📎 근거: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf, csv_me...
  검색: 8개 | 1.8초

[q02] 봉화군 재난통합관리시스템 고도화 사업 예산을 알려줘
  ✓ 답변: - 봉화군 재난통합관리시스템 고도화 사업의 예산은 900,000,000원입니다.

📎 근거: [CSV] - 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hw...
  검색: 8개 | 2.9초

[q03] 서울특별시 지도정보 플랫폼 사업의 평가기준을 알려줘
  ✓ 답변: 서울특별시 지도정보 플랫폼 사업의 평가기준은 다음과 같습니다:

- **제안서 평가**: 
  - 기술능력 평가: 90점
  - 입찰가격 평가: 10점
- **평가항목**:
  -...
  검색: 8개 | 23.2초

[q04] 한국철도공사 모바일오피스 시스템 고도화 용역의 입찰 참가자격은?
  ✓ 답변: 문서에서 한국철도공사 모바일오피스 시스템 고도화 용역의 입찰 참가자격에 대한 정보는 확인되지 않습니다. 

📎 근거: 한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및...
  검색: 8개 | 2.2초

[q05] 인천광역시 도시계획위원회 사업기간은?
  ✓ 답변: - 인천광역시 도시계획위원회 통합관리시스템 구축 용역의 사업 기간은 착수일로부터 180일입니다. 
- 사업비는 150,000,000원 (VAT 포함)입니다.

📎 근거: [근거 3...
  검색: 8개 | 2.5초

[q06] 을지대학교 비교과시스템 개발 사업 예산은?
  ✓ 답변: - 을지대학교 비교과시스템 개발 사업의 예산은 0.0입니다.

📎 근거: 을지대학교_을지대학교 비교과시스템 개발.hwp, csv_meta...
  검색: 8개 | 2.7초

[q07] 보험개발원 실손보험 청구 전산화 시스템의 제출서류는 무엇인가요?
  ✓

In [161]:
# ============================================================
# 셀 13: LLM 저지 평가 (4대 지표 × 객관식 라벨링)
# ============================================================
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import json
import re
 
EVAL_RESULTS_PATH = ARTIFACTS / "eval_results.json"
 
# LLM 저지 (평가 전용 모델)
judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,  # 일관성 최대
    max_tokens=500,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)
 
# ── 4대 지표별 객관식 프롬프트 ─────────────────────────
 
JUDGE_PROMPTS = {
    "context_relevance": {
        "system": """당신은 RAG 시스템의 검색 품질을 평가하는 심판입니다.
질문에 답하기 위해 검색된 문서(Context)가 실제로 관련 있는지 평가합니다.
 
반드시 아래 보기 중 하나만 선택하고, JSON 형식으로만 응답하세요.
다른 텍스트는 절대 포함하지 마세요.
 
보기:
1: 검색 결과가 질문과 매우 관련 있고, 답변에 필요한 정보를 포함함
2: 검색 결과가 부분적으로 관련 있으나, 핵심 정보가 부족함
3: 검색 결과가 질문과 거의 관련 없음
4: 검색 결과가 전혀 없거나 완전히 무관함""",
        "template": """[질문] {question}
[검색된 문서 내용]
{context}
[정답 섹션] {gold_section}
 
위 검색 결과가 질문에 답하기에 관련 있는지 평가하세요.
응답 형식: {{"label": 숫자, "reason": "한줄 이유"}}"""
    },
 
    "groundedness": {
        "system": """당신은 RAG 시스템의 근거성을 평가하는 심판입니다.
생성된 답변이 검색된 문서에 근거하고 있는지 평가합니다.
문서에 없는 내용을 지어냈는지(환각) 확인합니다.
 
반드시 아래 보기 중 하나만 선택하고, JSON 형식으로만 응답하세요.
 
보기:
1: 답변의 모든 내용이 검색 문서에 근거함 (환각 없음)
2: 답변 대부분이 근거하나, 일부 추측이 포함됨
3: 답변에 문서에 없는 내용이 상당히 포함됨 (환각 있음)
4: 답변이 완전히 지어낸 내용이거나, 문서와 무관함""",
        "template": """[질문] {question}
[검색된 문서 내용]
{context}
[생성된 답변]
{answer}
 
답변이 검색 문서에 근거하는지 평가하세요.
응답 형식: {{"label": 숫자, "reason": "한줄 이유"}}"""
    },
 
    "answer_relevance": {
        "system": """당신은 RAG 시스템의 답변 적합성을 평가하는 심판입니다.
생성된 답변이 사용자의 질문 의도에 정확히 부합하는지 평가합니다.
 
반드시 아래 보기 중 하나만 선택하고, JSON 형식으로만 응답하세요.
 
보기:
1: 질문에 정확히 답변하며, 필요한 정보가 충분함
2: 질문에 대략 답변하나, 핵심 정보가 일부 빠짐
3: 질문과 관련은 있으나, 의도에 맞지 않는 답변
4: 질문과 무관한 답변이거나 답변 실패""",
        "template": """[질문] {question}
[정답(참고)] {gold_answer}
[생성된 답변]
{answer}
 
생성된 답변이 질문의 의도에 부합하는지 평가하세요.
응답 형식: {{"label": 숫자, "reason": "한줄 이유"}}"""
    },
 
    "faithfulness": {
        "system": """당신은 RAG 시스템의 충실성을 평가하는 심판입니다.
생성된 답변이 원본 문서의 내용을 왜곡하지 않고 충실하게 전달하는지 평가합니다.
 
반드시 아래 보기 중 하나만 선택하고, JSON 형식으로만 응답하세요.
 
보기:
1: 문서 내용을 정확하고 충실하게 전달함
2: 대체로 충실하나, 표현이 약간 부정확함
3: 문서 내용을 상당히 왜곡하거나 과장함
4: 문서와 완전히 다른 내용을 전달함""",
        "template": """[질문] {question}
[검색된 문서 내용]
{context}
[정답(참고)] {gold_answer}
[생성된 답변]
{answer}
 
답변이 문서 내용을 충실하게 전달하는지 평가하세요.
응답 형식: {{"label": 숫자, "reason": "한줄 이유"}}"""
    },
 
    "abstention": {
        "system": """당신은 RAG 시스템의 답변 거부 능력을 평가하는 심판입니다.
문서에 답이 없는 질문에 대해 시스템이 올바르게 "모르겠다"고 답했는지 평가합니다.
 
반드시 아래 보기 중 하나만 선택하고, JSON 형식으로만 응답하세요.
 
보기:
1: 정확하게 "문서에서 확인되지 않음" 또는 "잘 모르겠습니다"라고 답변함
2: 답이 없다는 뉘앙스를 전달했으나, 일부 추측을 덧붙임
3: 답이 없는데도 자신있게 답변을 생성함 (환각)
4: 질문 자체를 이해하지 못한 듯한 답변""",
        "template": """[질문] {question}
[이 질문은 문서에 답이 없는 질문입니다]
[생성된 답변]
{answer}
 
시스템이 "답이 없다"는 것을 올바르게 인식했는지 평가하세요.
응답 형식: {{"label": 숫자, "reason": "한줄 이유"}}"""
    },
}
 
 
def parse_judge_response(response_text: str) -> dict:
    """LLM 저지 응답을 파싱"""
    try:
        # JSON 추출
        json_match = re.search(r'\{[^}]+\}', response_text)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass
    return {"label": -1, "reason": "파싱 실패"}
 
 
def evaluate_single(prediction: dict, metric: str) -> dict:
    """단일 예측에 대해 하나의 지표를 LLM 저지로 평가"""
    prompt_config = JUDGE_PROMPTS[metric]
 
    # Context 구성
    context = ""
    for doc in prediction.get("retrieved_docs", [])[:5]:
        context += f"[{doc['section_label']}] {doc['content_preview']}\n\n"
 
    # 프롬프트 채우기
    user_msg = prompt_config["template"].format(
        question=prediction["question"],
        context=context,
        answer=prediction.get("generated_answer", ""),
        gold_answer=prediction.get("gold_answer", ""),
        gold_section=prediction.get("gold_section", ""),
    )
 
    try:
        response = judge_llm.invoke([
            SystemMessage(content=prompt_config["system"]),
            HumanMessage(content=user_msg),
        ])
        result = parse_judge_response(response.content)
        return result
    except Exception as e:
        return {"label": -1, "reason": f"오류: {e}"}
 
 
def run_evaluation(predictions: list) -> list:
    """전체 예측에 대해 4대 지표 + 거부 평가 실행"""
    eval_results = []
 
    for pred in predictions:
        qid = pred["question_id"]
        print(f"\n[{qid}] 평가 중...")
 
        result = {
            "question_id": qid,
            "question": pred["question"],
            "question_type": pred["question_type"],
            "generated_answer": pred["generated_answer"][:200],
            "gold_answer": pred["gold_answer"],
            "num_retrieved": pred["num_retrieved"],
            "latency_sec": pred["latency_sec"],
        }
 
        # 답변 가능한 질문: 4대 지표 평가
        if pred["is_answerable"]:
            for metric in ["context_relevance", "groundedness", "answer_relevance", "faithfulness"]:
                judge_result = evaluate_single(pred, metric)
                result[f"{metric}_label"] = judge_result.get("label", -1)
                result[f"{metric}_reason"] = judge_result.get("reason", "")
                print(f"  {metric}: {judge_result.get('label', '?')} - {judge_result.get('reason', '')[:50]}")
        else:
            # 답변 불가 질문: 거부 능력 평가
            judge_result = evaluate_single(pred, "abstention")
            result["abstention_label"] = judge_result.get("label", -1)
            result["abstention_reason"] = judge_result.get("reason", "")
            print(f"  abstention: {judge_result.get('label', '?')} - {judge_result.get('reason', '')[:50]}")
 
            # 나머지 지표는 N/A
            for metric in ["context_relevance", "groundedness", "answer_relevance", "faithfulness"]:
                result[f"{metric}_label"] = "N/A"
                result[f"{metric}_reason"] = "답변 불가 질문"
 
        eval_results.append(result)
 
    # 저장
    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, ensure_ascii=False, indent=2)
    print(f"\n✓ 평가 결과 저장: {EVAL_RESULTS_PATH}")
 
    return eval_results
 
 
# 실행
eval_results = run_evaluation(predictions)


[q01] 평가 중...
  context_relevance: 3 - 검색 결과에 예산에 대한 정보가 전혀 포함되어 있지 않음
  groundedness: 4 - 문서에 예산에 대한 정보가 포함되어 있지 않음
  answer_relevance: 1 - 질문에 정확히 답변하며, 필요한 정보가 충분함
  faithfulness: 1 - 문서 내용을 정확하고 충실하게 전달함

[q02] 평가 중...
  context_relevance: 1 - 검색 결과에 봉화군 재난통합관리시스템 고도화 사업의 예산 정보가 포함되어 있음
  groundedness: 3 - 답변에 문서에 없는 내용이 상당히 포함됨 (환각 있음)
  answer_relevance: 1 - 질문에 정확히 답변하며, 필요한 정보가 충분함
  faithfulness: 1 - 문서 내용을 정확하고 충실하게 전달함

[q03] 평가 중...
  context_relevance: 1 - 검색 결과가 서울특별시 지도정보 플랫폼 사업의 평가기준에 대한 구체적인 정보를 포함하고 있
  groundedness: 1 - 답변의 모든 내용이 검색 문서에 근거함 (환각 없음)
  answer_relevance: 1 - 질문에 정확히 답변하며, 필요한 정보가 충분히 제공됨
  faithfulness: 1 - 문서 내용을 정확하고 충실하게 전달함

[q04] 평가 중...
  context_relevance: 2 - 입찰 참가자격에 대한 정보가 일부 포함되어 있으나, 핵심 정보가 부족함
  groundedness: 1 - 답변은 문서에서 확인된 내용을 정확히 반영하고 있음
  answer_relevance: 4 - 질문에 대한 답변이 전혀 제공되지 않고, 관련 정보가 누락됨
  faithfulness: 1 - 문서에서 입찰 참가자격에 대한 정보가 확인되지 않았음을 정확하게 전달함

[q05] 평가 중...
  context_relevance: 1 - 검색 결과에 도시계획위원회의 사업기간에 대한 구체적인 정보가 포함되어 있음


In [162]:
# ============================================================
# 셀 14: 평가 결과 분석 + 리포트
# ============================================================
import pandas as pd
 
def generate_eval_report(eval_results: list):
    """평가 결과를 분석하여 리포트 생성"""
 
    df = pd.DataFrame(eval_results)
    metrics = ["context_relevance", "groundedness", "answer_relevance", "faithfulness"]
 
    print("=" * 70)
    print("  📊 입찰메이트 RAG 평가 리포트")
    print("=" * 70)
 
    # ── 전체 요약 ────────────────────────────────────
    answerable = df[df["question_type"] != "unanswerable"]
    unanswerable = df[df["question_type"] == "unanswerable"]
 
    print(f"\n총 질문: {len(df)}개")
    print(f"  답변 가능: {len(answerable)}개")
    print(f"  답변 불가: {len(unanswerable)}개")
    print(f"  평균 응답 시간: {df['latency_sec'].mean():.1f}초")
 
    # ── 4대 지표 점수 (답변 가능 질문만) ───────────────
    print(f"\n{'─'*50}")
    print("  📈 4대 지표 평가 (1=최고, 4=최저)")
    print(f"{'─'*50}")
 
    for metric in metrics:
        col = f"{metric}_label"
        valid = answerable[answerable[col] != "N/A"][col].astype(int)
        if len(valid) == 0:
            continue
 
        avg = valid.mean()
        perfect = (valid == 1).sum()
        partial = (valid == 2).sum()
        bad = (valid >= 3).sum()
 
        bar_len = 20
        perfect_bar = "█" * int(perfect / len(valid) * bar_len)
        partial_bar = "▓" * int(partial / len(valid) * bar_len)
        bad_bar = "░" * (bar_len - len(perfect_bar) - len(partial_bar))
 
        print(f"\n  {metric}:")
        print(f"    평균: {avg:.2f} | 완벽(1): {perfect} | 부분(2): {partial} | 미흡(3+): {bad}")
        print(f"    [{perfect_bar}{partial_bar}{bad_bar}] {perfect}/{len(valid)} 완벽")
 
    # ── 거부 능력 (unanswerable만) ────────────────────
    if len(unanswerable) > 0:
        print(f"\n{'─'*50}")
        print("  🚫 답변 거부 능력 (unanswerable)")
        print(f"{'─'*50}")
        abstention_scores = unanswerable["abstention_label"]
        valid_abs = abstention_scores[abstention_scores != "N/A"].astype(int)
        if len(valid_abs) > 0:
            correct_refusal = (valid_abs == 1).sum()
            print(f"    정확한 거부: {correct_refusal}/{len(valid_abs)}")
            print(f"    거부 정확도: {correct_refusal/len(valid_abs)*100:.0f}%")
 
    # ── 질문 유형별 분석 ──────────────────────────────
    print(f"\n{'─'*50}")
    print("  📋 질문 유형별 분석")
    print(f"{'─'*50}")
 
    for qtype in ["single_doc", "multi_doc", "unanswerable"]:
        subset = df[df["question_type"] == qtype]
        if len(subset) == 0:
            continue
        print(f"\n  [{qtype}] ({len(subset)}개)")
        for _, row in subset.iterrows():
            answer_preview = str(row["generated_answer"])[:60]
            scores = []
            for m in metrics:
                lbl = row.get(f"{m}_label", "N/A")
                if lbl != "N/A":
                    scores.append(f"{m[:3]}={lbl}")
            if row.get("abstention_label") not in [None, "N/A"]:
                scores.append(f"abs={row['abstention_label']}")
            score_str = " | ".join(scores) if scores else ""
            print(f"    {row['question_id']}: {score_str}")
            print(f"      Q: {row['question'][:50]}")
            print(f"      A: {answer_preview}")
 
    # ── Retrieval 분석 ────────────────────────────────
    print(f"\n{'─'*50}")
    print("  🔍 Retrieval 분석")
    print(f"{'─'*50}")
    print(f"    평균 검색 chunk 수: {df['num_retrieved'].mean():.1f}")
 
    # ── 개선 포인트 ───────────────────────────────────
    print(f"\n{'─'*50}")
    print("  💡 개선 포인트")
    print(f"{'─'*50}")
 
    for _, row in answerable.iterrows():
        issues = []
        for m in metrics:
            lbl = row.get(f"{m}_label", "N/A")
            if lbl != "N/A" and int(lbl) >= 3:
                issues.append(m)
        if issues:
            print(f"    ⚠️ {row['question_id']}: {', '.join(issues)} 미흡")
            for m in issues:
                print(f"       → {m}: {row.get(f'{m}_reason', '')[:60]}")
 
    print(f"\n{'='*70}")
 
    return df
 
 
# 실행
eval_df = generate_eval_report(eval_results)
 
# CSV로도 저장
eval_csv_path = ARTIFACTS / "eval_results.csv"
eval_df.to_csv(eval_csv_path, index=False, encoding="utf-8-sig")
print(f"\n✓ CSV 저장: {eval_csv_path}")

  📊 입찰메이트 RAG 평가 리포트

총 질문: 12개
  답변 가능: 10개
  답변 불가: 2개
  평균 응답 시간: 5.1초

──────────────────────────────────────────────────
  📈 4대 지표 평가 (1=최고, 4=최저)
──────────────────────────────────────────────────

  context_relevance:
    평균: 1.60 | 완벽(1): 6 | 부분(2): 2 | 미흡(3+): 2
    [████████████▓▓▓▓░░░░] 6/10 완벽

  groundedness:
    평균: 2.10 | 완벽(1): 5 | 부분(2): 0 | 미흡(3+): 5
    [██████████░░░░░░░░░░] 5/10 완벽

  answer_relevance:
    평균: 1.60 | 완벽(1): 7 | 부분(2): 1 | 미흡(3+): 2
    [██████████████▓▓░░░░] 7/10 완벽

  faithfulness:
    평균: 1.50 | 완벽(1): 7 | 부분(2): 1 | 미흡(3+): 2
    [██████████████▓▓░░░░] 7/10 완벽

──────────────────────────────────────────────────
  🚫 답변 거부 능력 (unanswerable)
──────────────────────────────────────────────────
    정확한 거부: 2/2
    거부 정확도: 100%

──────────────────────────────────────────────────
  📋 질문 유형별 분석
──────────────────────────────────────────────────

  [single_doc] (8개)
    q01: con=3 | gro=4 | ans=1 | fai=1 | abs=nan
      Q: 고려대학교 차세대 포털 사업의 예산은 얼마인가요?
    